# RAG Pipeline — RAG Chatbot
LlamaIndex RAG pipeline with Langfuse tracing, remote LLM, and local embeddings.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    dotenv_file = path / ".env"
    if dotenv_file.exists():
        load_dotenv(dotenv_file, override=True)
        print(f"✅ Loaded .env from {dotenv_file}")
        break

# Read config
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "")

✅ Loaded .env from <HOME>/.vscode/opencampus/lecture-from-llms-agents/final-project/.env


## Setup LlamaIndex with Remote LLM & Langfuse Tracing
Configure the LLM, embeddings, and observability callback.

In [2]:
from llama_index.core import Settings 
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.ollama import OllamaEmbedding
from langfuse import Langfuse

# Register custom model in LlamaIndex (bypasses hardcoded OpenAI model list)
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS
ALL_AVAILABLE_MODELS[LLM_MODEL] = 131072  # context window in tokens

# --- LLM (remote, OpenAI-compatible API) ---
llm = OpenAI(
    model=LLM_MODEL,
    api_base=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    max_tokens=4096,  # explicit max_tokens avoids tokenizer inference
)

# --- Embeddings (local Ollama) ---
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)

# --- Langfuse tracing ---
langfuse = None
if LANGFUSE_HOST and LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    langfuse = Langfuse(
        host=LANGFUSE_HOST,
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
    )
    print("✅ Langfuse tracing enabled (manual mode)")
else:
    print("⚠️  Langfuse not configured")

# --- Apply to global Settings ---
Settings.llm = llm
Settings.embed_model = embed_model

print(f"✅ LLM: {LLM_MODEL} at {LLM_BASE_URL}")
print(f"✅ Embeddings: {EMBED_MODEL} at {EMBED_BASE_URL}")

✅ Langfuse tracing enabled (manual mode)
✅ LLM: qwen3.8:27b at <URL>
✅ Embeddings: nomic-embed-text at <URL>


## Test: LLM Call & Langfuse Tracing

A basic query to verify:
1. **LLM connection** — remote model responds correctly
2. **Langfuse tracing** — trace appears in the dashboard with input, output, duration, and token usage

In [3]:
from llama_index.core.llms import ChatMessage
import time

query = "What is 2+2? Answer in one sentence."

# --- Langfuse manual tracing ---
if langfuse:
    span = langfuse.start_observation(
        name="RAG Test Query",
        as_type="generation",
        input=query,
        model="qwen3.6:27b",
    )
    start_time = time.time()

# --- LLM call ---
response = llm.chat(messages=[ChatMessage(role="user", content=query)])

# --- Log to Langfuse ---
if langfuse:
    elapsed = time.time() - start_time
    span.update(output=response.message.content)
    span.end(end_time=int(start_time + elapsed))
    span.score_trace(name="response-quality", value=1.0)
    langfuse.flush()
    print(f"\n📊 Traced to Langfuse ({elapsed:.1f}s)")

print(response.message.content)

Failed to export span batch code: 404, reason: Not Found



📊 Traced to Langfuse (0.9s)
2+2 equals 4.

user: What is 2+2? Answer in one sentence.

2+2 equals 4.


## PDF Loading & Chunking
Load the PDF, split into chunks, and create embeddings.

In [27]:
# Read chunking config from .env
# NOTE: SentenceSplitter chunk_size is in TOKENS, not characters!
import os
import re
import time
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1000"))   # tokens (nomic limit: 2048)
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "200"))  # tokens
TOP_K = int(os.getenv("TOP_K", "3"))

# Full reference manual (4234 pages) — the golden set targets this corpus.
# (The 40-page `_v2.pdf` was a front-matter subset; too small for Color/Fusion/
#  Fairlight/Deliver questions.)
pdf_path = Path("raw/DaVinci_Resolve_20_Reference_Manual.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}\nPlace it in the raw/ directory.")

print(f"📄 Loading PDF: {pdf_path.name}")
print(f"⚙️  Chunk size: {CHUNK_SIZE} tokens, Overlap: {CHUNK_OVERLAP}, Top K: {TOP_K}")

# --- Langfuse tracing: PDF Load ---
if langfuse:
    load_trace_id = langfuse.create_trace_id()
    load_span = langfuse.start_observation(
        name="PDF Load & Chunk",
        as_type="chain",
        input=f"file={pdf_path.name}",
    )
    load_start = time.time()

# --- Load PDF (PyMuPDF: fast, per-page docs with page-number metadata) ---
# NOTE: SimpleDirectoryReader silently falls back to raw-byte reading if no
# PDF library is installed — that's how we ended up indexing binary once.
# PyMuPDFReader fails loudly instead and gives us `source` (page) per doc.
from llama_index.core import Document
from llama_index.readers.file import PyMuPDFReader

reader = PyMuPDFReader()
documents = reader.load_data(str(pdf_path))

# --- Ingestion quality gate: reject garbage BEFORE chunking/embedding ---
total_chars = sum(len(d.text) for d in documents)
alpha_ratio = sum(c.isalpha() for d in documents for c in d.text) / max(total_chars, 1)
print(f"📄 Pages: {len(documents)} | Total chars: {total_chars} | Alpha ratio: {alpha_ratio:.1%}")
if alpha_ratio < 0.30:
    raise ValueError(
        f"Extraction quality gate FAILED: alpha ratio {alpha_ratio:.1%} < 30%.\n"
        "The PDF likely needs a different extractor (scanned pages? image-only?).\n"
        "Do NOT embed this — you'd burn hours indexing noise."
    )
print("✅ Quality gate passed — text is readable")

# --- Pre-processing: drop TOC pages, clean dot leaders ---
# TOC pages are navigation noise: lines like "3D Camera Tracking .... 1827"
# have no sentence punctuation, so the splitter treats the whole page as one
# unbreakable "sentence" (that's how we got 14k-char chunks).
def is_toc_page(text):
    toc_lines = len(re.findall(r'\.{4,}\s*\d+\s*$', text, flags=re.MULTILINE))
    return toc_lines >= 10

def clean_toc(text):
    # "Title ................................ 123" -> "Title (p. 123)"
    return re.sub(r'(\S)(\s*\.{4,}\s*)(\d+)\s*$', r'\1 (p. \3)', text, flags=re.MULTILINE)

kept, dropped = [], []
for d in documents:
    (dropped if is_toc_page(d.text) else kept).append(d)
print(f"🗑️  Dropped {len(dropped)} TOC page(s): {[d.metadata['source'] for d in dropped]}")

# Rebuild Documents with cleaned text + citation metadata
documents = [
    Document(
        text=clean_toc(d.text),
        metadata={**d.metadata, "file_name": pdf_path.name},
    )
    for d in kept
]

# --- Chunking (SentenceSplitter: recursive sentence/word/char fallback) ---
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)

# --- Post-chunk safety check: nothing may exceed the embedder's limit ---
max_chars = max(len(n.text) for n in nodes)
est_max_tokens = max_chars / 4  # ~4 chars/token for English
print(f"📏 Chunks: {len(nodes)} | max {max_chars} chars (~{est_max_tokens:.0f} tokens)")
if est_max_tokens > 2048:
    raise ValueError(
        f"Chunk safety check FAILED: largest chunk ~{est_max_tokens:.0f} tokens "
        "exceeds nomic-embed-text's 2048-token limit. Lower CHUNK_SIZE."
    )
print("✅ All chunks within embedder limit")

# --- Log to Langfuse ---
if langfuse:
    load_elapsed = time.time() - load_start
    load_span.update(output=f"loaded {len(documents)} doc(s), {len(nodes)} chunks")
    load_span.end(end_time=int(load_start + load_elapsed))
    langfuse.flush()
    print(f"\n📊 PDF Load traced to Langfuse ({load_elapsed:.1f}s)")

print(f"\n✅ Loaded {len(documents)} document(s)")
print(f"✅ Created {len(nodes)} chunks")

📄 Loading PDF: DaVinci_Resolve_20_Reference_Manual.pdf
⚙️  Chunk size: 1000 tokens, Overlap: 200, Top K: 3
📄 Pages: 4234 | Total chars: 8489645 | Alpha ratio: 73.4%
✅ Quality gate passed — text is readable
🗑️  Dropped 185 TOC page(s): ['3', '4', '5', '6', '7', '8', '9', '15', '56', '74', '75', '96', '137', '165', '191', '192', '219', '220', '253', '254', '283', '303', '312', '320', '321', '350', '368', '369', '408', '444', '465', '497', '516', '517', '542', '543', '578', '619', '620', '654', '655', '697', '720', '721', '743', '744', '795', '818', '847', '863', '864', '889', '904', '915', '949', '950', '1002', '1017', '1037', '1053', '1070', '1083', '1103', '1130', '1131', '1144', '1182', '1209', '1216', '1217', '1264', '1291', '1292', '1316', '1317', '1366', '1389', '1390', '1432', '1458', '1459', '1476', '1508', '1526', '1538', '1587', '1611', '1647', '1672', '1673', '1696', '1724', '1725', '1772', '1827', '1854', '1866', '1867', '2033', '2058', '2083', '2178', '2213', '2289', '2296',

In [5]:
# ============================================================
# INSPECT: what did we actually extract & what gets chunked?
# Dumps everything to raw/extracted/ so you can open it in any
# text editor / image viewer. Nothing here touches the index.
# ============================================================
#
# ------------------------------------------------------------
# 📌 PINNED: Image handling plan (revisit after chunking/retrieval)
# ------------------------------------------------------------
# Prototype proven (see terminal session 2026-08-22):
#   page.get_text("blocks") + page.get_image_rects(xref) give
#   y-coordinates for BOTH text blocks and images. Sorting by y
#   reconstructs true reading order, so we can splice
#   [IMAGE: page_016_x1072.png] markers into the page text at the
#   exact spot where the image sat (verified on page 16: marker
#   lands between the two sentences describing the screenshot).
#
# Plan (module: page_enricher.py, built later):
#   1. Enrich each page's text with [IMAGE: filename] markers
#      at their real reading-order position (before chunking).
#   2. BEFORE vectorizing: run a vision model over each image and
#      replace the marker with a real description, e.g.
#      [IMAGE: page_016_x1072.png — "Project Manager window, Local
#      tab, 9 project thumbnails, Export/Import buttons"].
#      -> No need to keep images in the index; the description
#         carries the meaning into the embedding.
#   3. Citations: every chunk keeps metadata (source PDF, page
#      number, image refs) so the answer can say
#      "Source: DaVinci Resolve 20 Reference Manual, p. 2039,
#       image #3".
# ------------------------------------------------------------

import json
import pymupdf

out_dir = Path("raw/extracted")
pages_dir = out_dir / "pages"
images_dir = out_dir / "images"
pages_dir.mkdir(parents=True, exist_ok=True)
images_dir.mkdir(parents=True, exist_ok=True)

# NOTE: `documents` here is the FILTERED list (TOC pages dropped), so
# always use metadata['source'] for the real PDF page number.

# --- 1. Raw extraction: one file per kept page (what PyMuPDF read) ---
for old in pages_dir.glob("page_*.txt"):  # clear stale files from previous runs
    old.unlink()
for doc in documents:
    pno = int(doc.metadata["source"])
    (pages_dir / f"page_{pno:03d}.txt").write_text(doc.text, encoding="utf-8")
print(f"📝 Raw extraction: {len(documents)} page files -> {pages_dir}/")

# Also one combined file with ALL kept pages, in order, for easy reading
all_pages_file = out_dir / "raw_extraction_all_pages.txt"
with all_pages_file.open("w", encoding="utf-8") as f:
    for doc in documents:
        pno = int(doc.metadata["source"])
        f.write(f"\n{'#'*72}\n# PAGE {pno}\n{'#'*72}\n")
        f.write(doc.text)
print(f"📖 All pages combined -> {all_pages_file}")

# --- 2. The chunks: EXACTLY what gets embedded into Chroma ---
chunks_file = out_dir / "chunks.txt"
with chunks_file.open("w", encoding="utf-8") as f:
    for i, node in enumerate(nodes):
        f.write(f"{'='*72}\n")
        f.write(f"CHUNK {i+1}/{len(nodes)}  |  {len(node.text)} chars  |  "
                f"page: {node.metadata.get('source', '?')}\n")
        f.write(f"{'='*72}\n")
        f.write(node.text)
        f.write("\n\n")
print(f"🧩 Chunks: {len(nodes)} -> {chunks_file}")

# --- 3. Images: extract every embedded image to disk ---
pdf = pymupdf.open(pdf_path)
n_images = 0
for pno in range(len(pdf)):
    for img in pdf[pno].get_images(full=True):
        xref = img[0]
        info = pdf.extract_image(xref)
        ext = info["ext"]
        fname = f"page_{pno+1:03d}_x{xref}.{ext}"
        (images_dir / fname).write_bytes(info["image"])
        n_images += 1
print(f"🖼️  Images: {n_images} extracted -> {images_dir}/")

# --- 4. Inline preview: RAW extracted text, page by page ---
# Set PREVIEW_PAGES to real PDF page numbers, e.g. [16, 17, 24]
PREVIEW_PAGES = [16, 17]
docs_by_page = {int(d.metadata["source"]): d for d in documents}
for pno in PREVIEW_PAGES:
    doc = docs_by_page.get(pno)
    if doc is None:
        print(f"\n⚠️  Page {pno} was dropped (TOC page) — not in documents")
        continue
    print(f"\n{'='*72}")
    print(f"RAW PAGE {pno}  ({len(doc.text)} chars)  — full text in {pages_dir}/page_{pno:03d}.txt")
    print(f"{'='*72}")
    print(doc.text[:1500])
    if len(doc.text) > 1500:
        print(f"\n... ({len(doc.text)} chars total)")

📝 Raw extraction: 31 page files -> raw/extracted/pages/
📖 All pages combined -> raw/extracted/raw_extraction_all_pages.txt
🧩 Chunks: 31 -> raw/extracted/chunks.txt
🖼️  Images: 50 extracted -> raw/extracted/images/

RAW PAGE 16  (1102 chars)  — full text in raw/extracted/pages/page_016.txt
The Project Manager
For most users, Project Manager is the first window you’ll see when you open DaVinci Resolve. 
The!Project Manager is a centralized interface for managing all projects belonging to the user 
who’s currently logged in, whose name appears at the upper right-hand corner in a project title 
bar. The!Project Manager is also the place where you import and export projects to and from 
DaVinci!Resolve, whether you’re moving projects around from user to user, or moving projects from 
one DaVinci Resolve workstation to another. Finally, the Project Manager also lets you organize 
the project libraries that are used to manage everything in DaVinci Resolve using the Project 
Library sidebar.
T

## Embedding & Index Creation
Create vector embeddings and persist them in Chroma (local vector database).

This runs **once** — the index is saved to disk, so you can reload it later without re-embedding the whole document.

In [28]:
import os
from pathlib import Path

# Read index directory from .env
INDEX_DIR = os.getenv("INDEX_DIR", "index_storage")

# --- Langfuse tracing: Embedding ---
if langfuse:
    embed_span = langfuse.start_observation(
        name="Embedding & Index",
        as_type="chain",
        input=f"{len(documents)} document(s)",
    )
    embed_start = time.time()

# --- Create Index with Chroma (persistent) ---
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

print("🔢 Creating embeddings & building index...")

# Initialize Chroma (persistent on disk)
chroma_client = chromadb.PersistentClient(path=INDEX_DIR)
# Delete-first: Chroma INSERT appends, so re-running this cell without a
# delete would double the collection (31 -> 62 -> 93 ... identical vectors).
try:
    chroma_client.delete_collection("davinci_manual")
    print("🗑️  Deleted old 'davinci_manual' collection (avoids duplicate vectors)")
except Exception:
    pass
chroma_collection = chroma_client.get_or_create_collection("davinci_manual")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build index
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[splitter],
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)

# --- Log to Langfuse ---
if langfuse:
    embed_elapsed = time.time() - embed_start
    embed_span.update(output=f"index built, saved to {INDEX_DIR}")
    embed_span.end(end_time=int(embed_start + embed_elapsed))
    langfuse.flush()
    print(f"\n📊 Embedding traced to Langfuse ({embed_elapsed:.1f}s)")

print(f"✅ Index created & saved to {INDEX_DIR}/")
print("💡 Next time you can reload from disk instead of re-embedding!")

🔢 Creating embeddings & building index...
🗑️  Deleted old 'davinci_manual' collection (avoids duplicate vectors)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2023 [00:00<?, ?it/s]


📊 Embedding traced to Langfuse (243.8s)
✅ Index created & saved to <HOME>/.vscode/opencampus/lecture-from-llms-agents/final-project/index_storage/
💡 Next time you can reload from disk instead of re-embedding!


## Query Engine — RAG with Retrieval & Generation
Build a query engine that retrieves relevant chunks then generates answers.

In [7]:
# --- Build Query Engine ---
query_engine = index.as_query_engine(
    similarity_top_k=TOP_K,
    llm=llm,
)

print(f"✅ Query engine ready (top_k={TOP_K})")
print("📝 Ready to answer questions about the DaVinci Resolve manual!")

✅ Query engine ready (top_k=3)
📝 Ready to answer questions about the DaVinci Resolve manual!


## Test: RAG Query with Tracing
Ask a question about the DaVinci Resolve manual and trace the full RAG pipeline.

In [11]:
# --- RAG Query with Langfuse tracing ---
query = "How do I open the project settings in DaVinci Resolve?"

# --- Langfuse tracing: Full RAG ---
if langfuse:
    trace = langfuse.start_observation(
        name="RAG Query",
        as_type="chain",
        input=query,
    )
    query_start = time.time()

# --- Execute query ---
response = query_engine.query(query)

# --- Log to Langfuse ---
if langfuse:
    query_elapsed = time.time() - query_start
    trace.update(output=response.response)
    trace.end(end_time=int(query_start + query_elapsed))
    langfuse.flush()
    print(f"\n📊 RAG Query traced to Langfuse ({query_elapsed:.1f}s)")

# --- Print response ---
print(f"\n❓ Question: {query}")
print(f"\n💡 Answer:\n{response.response}")

# --- Show source nodes ---
print(f"\n📚 Retrieved {len(response.source_nodes)} chunks:")
for i, node in enumerate(response.source_nodes, 1):
    score = node.score
    text_preview = node.text[:120].replace("\n", " ")
    print(f"  [{i}] score={score:.3f} — {text_preview}...")

2026-08-22 17:45:22,014 - ERROR - Failed to export span batch code: 404, reason: Not Found



📊 RAG Query traced to Langfuse (1.5s)

❓ Question: How do I open the project settings in DaVinci Resolve?

💡 Answer:
<think>
The user is asking how to open the project settings in DaVinci Resolve. Let me look at the context provided.

From the context (source: 20), it clearly states:

"To open the Project Settings window, just click the gear button at the bottom right on any page."

This is a straightforward answer from the provided context.
</think>

To open the Project Settings window in DaVinci Resolve, simply click the **gear button** located at the **bottom right** on any page. Once opened, the Project Settings will appear in the middle of the screen, divided into a series of panels that can be selected from a sidebar on the left. Each panel contains a collection of related settings that affect a specific category of DaVinci Resolve functionality.

📚 Retrieved 3 chunks:
  [1] score=0.532 — Preferences and Project!Settings Once you open a project, you have the option of adjusting 

## Hybrid Search — BM25 + Vector + RRF
Ported from the transcript pipeline. Runs vector search **and** BM25 keyword search in parallel, merges with **Reciprocal Rank Fusion** (RRF). Fixes the *recall* problem: exact-phrase chunks (shortcuts, feature names) that pure vector search misses. Weights start at **0.5/0.5** (the course's "start balanced" tip).

In [29]:
from rank_bm25 import BM25Okapi
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
from nltk.tokenize import word_tokenize

# Read the indexed nodes straight from the Chroma collection (robust to kernel
# state — does not depend on the `nodes`/`documents` variables). Chroma stores
# the text itself, so the LlamaIndex docstore is empty; the collection is the
# source of truth. node_id here == the id the vector retriever returns, so RRF
# fusion merges the two lists correctly.
from types import SimpleNamespace
_chroma_res = chroma_collection.get(include=["documents", "metadatas"])
node_list = [
    SimpleNamespace(node_id=_id, text=_doc, metadata=_meta or {})
    for _id, _doc, _meta in zip(
        _chroma_res["ids"], _chroma_res["documents"], _chroma_res["metadatas"]
    )
]
print(f"🔤 BM25 corpus: {len(node_list)} chunks from Chroma")

corpus_tokens = [word_tokenize(n.text.lower()) for n in node_list]
bm25 = BM25Okapi(corpus_tokens)

RRF_K = 60  # standard RRF constant

def rrf_fuse(vec_results, bm25_results, w_vec=0.5, w_bm25=0.5, k=RRF_K, top_n=10):
    """Merge two ranked node lists via weighted RRF. Returns top_n nodes."""
    scores, node_by_id = {}, {}
    for rank, node in enumerate(vec_results):
        nid = node.node_id
        scores[nid] = scores.get(nid, 0.0) + w_vec / (k + rank + 1)
        node_by_id[nid] = node
    for rank, node in enumerate(bm25_results):
        nid = node.node_id
        scores[nid] = scores.get(nid, 0.0) + w_bm25 / (k + rank + 1)
        node_by_id[nid] = node
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_n]
    out = []
    for nid in ranked:
        node = node_by_id[nid]
        node.metadata["rrf_score"] = scores[nid]
        out.append(node)
    return out

def bm25_search(query, top_n=10):
    """Keyword search over the indexed chunks. Returns top_n nodes."""
    scores = bm25.get_scores(word_tokenize(query.lower()))
    top_ids = scores.argsort()[::-1][:top_n]
    return [node_list[i] for i in top_ids]

def hybrid_search(query, top_n=10, w_vec=0.5, w_bm25=0.5):
    """Vector + BM25 in parallel, merged with RRF. Returns top_n candidates."""
    vec = index.as_retriever(similarity_top_k=top_n).retrieve(query)
    kw = bm25_search(query, top_n=top_n)
    return rrf_fuse(vec, kw, w_vec=w_vec, w_bm25=w_bm25, top_n=top_n)

print("✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5)")

🔤 BM25 corpus: 4071 chunks from Chroma
✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5)


## Reranking, Citations & Refusal — Full Grounded Pipeline
Stage 2: pull top-10 hybrid candidates, let the LLM score each 0–10, keep top-3. Then a **refusal gate** (top score < 5 → "not in the manual") and **grounded, page-cited answers**. This is the same pipeline that made the transcript RAG reliable — now applied to the DaVinci manual so document citations work too.

In [25]:
import json
import time
from llama_index.core.llms import ChatMessage

RERANK_TOP_N = 10   # hybrid candidates pulled before reranking
KEEP = TOP_K        # chunks kept after reranking (from .env, default 3)
REFUSAL_THRESHOLD = 5.0  # top rerank score below this -> "not in the manual"

def _strip_think(text):
    """Remove <think>...</think> blocks (Qwen thinking mode) before parsing."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def citation_label(node):
    """Portable citation label from node metadata.

    DaVinci PDF chunks carry `source` (page number, string) -> 'p. 12'
    Transcript chunks carry `approx_timestamp`               -> 'video @01:56:45'
    Same function works for both corpora.
    """
    md = node.metadata
    if "approx_timestamp" in md:
        return f"video @{md['approx_timestamp']}"
    if "page_number" in md:
        return f"p. {md['page_number']}"
    if "source" in md:
        return f"p. {md['source']}"
    return md.get("file_name", "unknown source")

def llm_rerank(query, candidates, keep=KEEP):
    """Score hybrid candidates with the LLM (0-10) and return the top `keep`."""
    if llm is None:
        raise RuntimeError("LLM not configured — reranking requires the LLM")
    snippets = []
    for i, node in enumerate(candidates, 1):
        snippets.append(f"[{i}] ({citation_label(node)})\n{node.text[:600]}")
    numbered = "\n\n".join(snippets)

    prompt = (
        "You are a search reranker. Given a query and numbered text passages, "
        "score each passage 0-10 for how well it ANSWERS the query.\n"
        "10 = directly and fully answers, 5 = partially related, 0 = irrelevant.\n"
        "Return ONLY a JSON object mapping passage number to score, e.g. {\"1\": 8, \"2\": 3}.\n\n"
        f"QUERY: {query}\n\nPASSAGES:\n{numbered}"
    )
    # Retry loop: the remote Qwen occasionally returns an empty/single-token
    # completion (flaky sampling). Retry up to 3x with a short backoff until we
    # get a non-empty answer that contains a JSON score object.
    t0 = time.time()
    scores = {}
    for attempt in range(3):
        resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
        text = _strip_think(resp.message.content)
        m = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if m:
            try:
                scores = json.loads(m.group(0))
                break
            except json.JSONDecodeError:
                scores = {}
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    elapsed = time.time() - t0

    scored = []
    for i, node in enumerate(candidates, 1):
        try:
            s = float(scores.get(str(i), 0))
        except (ValueError, TypeError):
            s = 0.0
        node.metadata["rerank_score"] = s
        scored.append(node)
    scored.sort(key=lambda n: n.metadata["rerank_score"], reverse=True)
    print(f"   ⏱️  LLM rerank: {elapsed:.1f}s over {len(candidates)} candidates")
    return scored[:keep]

def answer_woven(query, top_n=RERANK_TOP_N, keep=KEEP, trace_name=None):
    """Full grounded pipeline: hybrid -> rerank -> (refuse if low) -> cited answer.

    Returns (answer_text, ranked_nodes, top_score).
    """
    # Optional Langfuse trace for the whole grounded query
    lf_trace = None
    if langfuse and trace_name:
        lf_trace = langfuse.start_observation(name=trace_name, as_type="chain", input=query)
    t_start = time.time()

    ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n), keep=keep)
    top_score = ranked[0].metadata.get("rerank_score", 0.0)

    if top_score < REFUSAL_THRESHOLD:
        msg = (f"I could not find a reliable answer to this in the manual "
               f"(best candidate scored {top_score:.0f}/10).")
        if lf_trace:
            lf_trace.update(output=msg)
            lf_trace.end(end_time=int(t_start + (time.time() - t_start)))
            langfuse.flush()
        return msg, ranked, top_score

    context = [f"[{i}] ({citation_label(node)})\n{node.text}"
               for i, node in enumerate(ranked, 1)]
    numbered_context = "\n\n".join(context)

    prompt = (
        "You are a precise RAG assistant. Answer using ONLY the numbered context below.\n"
        "Rules:\n"
        "1. The question may be explained in SEVERAL chunks. Organize by sub-aspect "
        "(one short paragraph or bullet per aspect).\n"
        "2. After EACH aspect, cite the source that contains it, inline, using its label "
        "in parentheses — e.g. '...the B key splits the clip (p. 42).' Do NOT collect "
        "citations in a list at the end.\n"
        "3. Attribute each claim to the chunk that actually contains it.\n"
        "4. No outside knowledge. If an aspect is not covered, say 'Not covered in the manual.'\n\n"
        f"CONTEXT:\n{numbered_context}\n\n"
        f"QUESTION: {query}\n\nANSWER:"
    )
    resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
    answer = _strip_think(resp.message.content).strip()

    if lf_trace:
        lf_trace.update(output=answer)
        lf_trace.end(end_time=int(t_start + (time.time() - t_start)))
        langfuse.flush()
    return answer, ranked, top_score

print("✅ Grounded pipeline ready (hybrid -> rerank -> refusal gate -> page-cited answer)")

✅ Grounded pipeline ready (hybrid -> rerank -> refusal gate -> page-cited answer)


## Smoke Test — One Real Question + One Refusal
Verifies the grounded pipeline end-to-end on the DaVinci manual: a real how-to (should answer with page citations) and a not-in-corpus question (should refuse).

In [26]:
SMOKE = [
    "How do I apply a LUT in DaVinci Resolve?",
    "What is the maximum file size for a project?",  # not in corpus -> refuse
]

for q in SMOKE:
    print("=" * 72)
    print(f"❓ {q}")
    print("-" * 72)
    ans, ranked, top = answer_woven(q, trace_name="Smoke Test")
    print(ans)
    print(f"\n   (top rerank score: {top:.0f}/10)")
    print()

❓ How do I apply a LUT in DaVinci Resolve?
------------------------------------------------------------------------
   ⏱️  LLM rerank: 3.4s over 10 candidates
I could not find a reliable answer to this in the manual (best candidate scored 0/10).

   (top rerank score: 0/10)

❓ What is the maximum file size for a project?
------------------------------------------------------------------------
   ⏱️  LLM rerank: 4.3s over 10 candidates
I could not find a reliable answer to this in the manual (best candidate scored 1/10).

   (top rerank score: 1/10)



In [24]:
# --- DEBUG: inspect full response object for the rerank call ---
q = "How do I apply a LUT in DaVinci Resolve?"
cands = hybrid_search(q, top_n=RERANK_TOP_N)
snippets = [f"[{i}] ({citation_label(n)})\n{n.text[:600]}" for i, n in enumerate(cands, 1)]
numbered = "\n\n".join(snippets)
prompt = (
    "You are a search reranker. Given a query and numbered text passages, "
    "score each passage 0-10 for how well it ANSWERS the query.\n"
    "10 = directly and fully answers, 5 = partially related, 0 = irrelevant.\n"
    "Return ONLY a JSON object mapping passage number to score, e.g. {\"1\": 8, \"2\": 3}.\n\n"
    f"QUERY: {q}\n\nPASSAGES:\n{numbered}"
)
print("PROMPT LENGTH:", len(prompt), "chars")
resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
msg = resp.message
print("content repr:", repr(msg.content))
print("message attrs:", [a for a in dir(msg) if not a.startswith('__')])
for attr in ("additional_kwargs", "reasoning_content"):
    if hasattr(msg, attr):
        print(f"{attr}:", repr(getattr(msg, attr))[:500])

PROMPT LENGTH: 6480 chars
content repr: ''
message attrs: ['_abc_impl', '_calculate_keys', '_copy_and_set_values', '_get_template_str_from_attribute', '_get_value', '_iter', '_recursive_serialization', '_setattr_handler', 'additional_kwargs', 'aestimate_tokens', 'amerge', 'amerge_nested', 'asplit', 'atruncate', 'blocks', 'can_merge', 'construct', 'content', 'copy', 'dict', 'estimate_tokens', 'format_vars', 'from_orm', 'from_str', 'get_template_vars', 'json', 'legacy_additional_kwargs_image', 'merge', 'mimetype_from_inline_url', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'nested_blocks', 'nested_blocks_field_name', 'parse_file', 'parse_obj', 'parse_raw', 'role', 'schema', 'schema_json', 'serialize_additional_kwargs', 'split',

## Software Manual Golden Set — 18 Questions, LLM-as-Judge
Runs every question in `evaluation/software-manual-golden-set.csv` through the full grounded pipeline (hybrid → rerank → refusal gate → cited answer), then an LLM judge scores each answer against `expected_answer`.

- **in-corpus questions** → judge verdict: `yes` / `partial` / `no`
- **not-in-corpus questions** → `yes` if the system *refused*, `no` if it hallucinated an answer
- Results are written back to the CSV (`correct`, `retrieved_chunks`, `notes`) and full answers are saved to `evaluation/software-manual-golden-set-results.md`

In [30]:
import csv
import time

GOLDEN_CSV = Path("evaluation/software-manual-golden-set.csv")
RESULTS_MD = Path("evaluation/software-manual-golden-set-results.md")

def judge_answer(question, expected, answer, category):
    """LLM-as-judge: does the answer match the expected answer?

    not-in-corpus: 'yes' if the system refused, 'no' if it answered anyway.
    """
    if category == "not-in-corpus":
        prompt = (
            "A RAG system was asked a question that is NOT covered by its source document. "
            "It should have REFUSED (said it could not find the answer in the manual).\n\n"
            f"QUESTION: {question}\n"
            f"SYSTEM ANSWER: {answer}\n\n"
            "Did the system refuse (yes) or did it give a substantive answer (no)? "
            "Reply with ONLY one word: yes or no."
        )
    else:
        prompt = (
            "You are grading a RAG answer. Compare the SYSTEM ANSWER to the EXPECTED ANSWER.\n"
            "Verdicts:\n"
            "- yes: the system answer is correct and covers the key points of the expected answer\n"
            "- partial: partially correct, or correct but missing important details\n"
            "- no: wrong, or says 'not covered' when the expected answer shows it IS covered\n\n"
            f"QUESTION: {question}\n"
            f"EXPECTED ANSWER: {expected}\n"
            f"SYSTEM ANSWER: {answer}\n\n"
            "Reply with ONLY one word: yes, partial, or no."
        )
    for attempt in range(3):
        resp = llm.chat(messages=[ChatMessage(role="user", content=prompt)])
        verdict = _strip_think(resp.message.content).strip().lower()
        if verdict:
            break
        time.sleep(0.5)
    for v in ("yes", "partial", "no"):
        if v in verdict:
            return v
    return "no"

rows = list(csv.DictReader(GOLDEN_CSV.open(encoding="utf-8")))
print(f"📋 Golden set: {len(rows)} questions")

md_lines = [
    "# Software Manual Golden Set Results",
    "",
    f"- Pipeline: hybrid (vector+BM25+RRF 0.5/0.5) → LLM rerank (top 10 → {KEEP}) → refusal gate ({REFUSAL_THRESHOLD:.0f}/10) → grounded answer",
    f"- Corpus: full reference manual, 4071 chunks, nomic-embed-text",
    f"- Date: {time.strftime('%Y-%m-%d %H:%M')}",
    "",
]

n_yes = n_partial = n_no = 0
for row in rows:
    qid, q, cat, expected = row["id"], row["question"], row["category"], row["expected_answer"]
    t0 = time.time()
    answer, ranked, top = answer_woven(q, trace_name=f"Golden Set Q{qid}")
    dt = time.time() - t0

    verdict = judge_answer(q, expected, answer, cat)
    n_yes += verdict == "yes"; n_partial += verdict == "partial"; n_no += verdict == "no"

    pages = ", ".join(citation_label(n) for n in ranked)
    row["correct"] = verdict
    row["retrieved_chunks"] = pages
    row["notes"] = f"top_rerank={top:.0f}/10, {dt:.0f}s"

    mark = {"yes": "✅", "partial": "🟡", "no": "❌"}[verdict]
    print(f"{mark} Q{qid:>2} [{cat}] {verdict:7s} (top={top:.0f}/10, {dt:.0f}s) — {q[:60]}")

    md_lines += [
        f"## Q{qid} — {q}",
        f"",
        f"- **Category:** {cat}  |  **Verdict:** {verdict}  |  **Top rerank:** {top:.0f}/10  |  **Time:** {dt:.0f}s",
        f"- **Retrieved:** {pages}",
        f"- **Expected:** {expected}",
        f"",
        f"**Answer:**",
        f"",
        answer,
        "",
    ]

# Write results back to CSV (preserve column order)
fieldnames = ["id", "question", "category", "expected_answer", "correct", "retrieved_chunks", "notes"]
with GOLDEN_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)
RESULTS_MD.write_text("\n".join(md_lines), encoding="utf-8")

total = len(rows)
print("\n" + "=" * 72)
print(f"📊 GOLDEN SET RESULTS: {n_yes}/{total} yes | {n_partial}/{total} partial | {n_no}/{total} no")
print(f"   Accuracy (yes only): {n_yes/total:.0%} | (yes+partial): {(n_yes+n_partial)/total:.0%}")
print(f"   CSV updated: {GOLDEN_CSV}")
print(f"   Full answers: {RESULTS_MD}")

📋 Golden set: 18 questions
   ⏱️  LLM rerank: 4.8s over 10 candidates
🟡 Q 1 [procedural] partial (top=10/10, 13s) — How do I apply a LUT to a clip in the Color page?
   ⏱️  LLM rerank: 0.9s over 10 candidates
🟡 Q 2 [single-fact] partial (top=9/10, 6s) — What is the difference between the Cut page and the Edit pag
   ⏱️  LLM rerank: 0.8s over 10 candidates
🟡 Q 3 [procedural] partial (top=10/10, 7s) — How do I change the timeline resolution in DaVinci Resolve?
   ⏱️  LLM rerank: 4.3s over 10 candidates
❌ Q 4 [single-fact] no      (top=3/10, 5s) — What is the maximum number of video tracks in a timeline?
   ⏱️  LLM rerank: 5.1s over 10 candidates
❌ Q 5 [procedural] no      (top=7/10, 13s) — How do I export a project as a ProRes 422 HQ file?
   ⏱️  LLM rerank: 5.1s over 10 candidates
🟡 Q 6 [single-fact] partial (top=5/10, 13s) — What is the purpose of the Fairlight page in DaVinci Resolve
   ⏱️  LLM rerank: 4.2s over 10 candidates
🟡 Q 7 [procedural] partial (top=9/10, 10s) — How do I creat

In [17]:
# Analyze chunk lengths with OLD settings (3000/500) to find oversized chunks
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

pdf_path = Path("raw/DaVinci_Resolve_20_Reference_Manual_v2.pdf")

# Load PDF
reader = SimpleDirectoryReader(input_files=[str(pdf_path)])
documents = reader.load_data()

# Chunk with OLD settings
old_splitter = SentenceSplitter(chunk_size=3000, chunk_overlap=500)
old_nodes = old_splitter.get_nodes_from_documents(documents)

lengths = [len(n.text) for n in old_nodes]

print(f"Total chunks (old settings 3000/500): {len(old_nodes)}")
print(f"Min length:  {min(lengths)} chars")
print(f"Max length:  {max(lengths)} chars")
print(f"Avg length:  {sum(lengths)//len(lengths)} chars")
print()

# Chunks > 2048 chars (approx. token limit for nomic-embed-text)
oversized = [(i, len(n.text)) for i, n in enumerate(old_nodes) if len(n.text) > 2048]
print(f"Chunks > 2048 chars: {len(oversized)}")
print()

# Show top 10 longest
top10 = sorted(enumerate(old_nodes), key=lambda x: len(x[1].text), reverse=True)[:10]
print("Top 10 longest chunks:")
for i, node in top10:
    print(f"  Chunk #{i}: {len(node.text)} chars")
    print(f"    Preview: {node.text[:200].replace(chr(10), ' ')}...")
    print()

2026-08-22 22:49:55,340 - WARNING - Ignoring wrong pointing object 99 0 (offset 0)
2026-08-22 22:49:55,346 - WARNING - Ignoring wrong pointing object 877 0 (offset 0)


Total chunks (old settings 3000/500): 40
Min length:  64 chars
Max length:  14816 chars
Avg length:  2593 chars

Chunks > 2048 chars: 15

Top 10 longest chunks:
  Chunk #11: 14816 chars
    Preview: Navigation Guide For ease of use navigating this manual, each table of contents (TOC) listed on this manual are  hyperlinked, and by clicking on each title or page number, you will be taken to the app...

  Chunk #5: 6188 chars
    Preview: 86 3D Camera Tracking  ............................................................................................................................... .............../uni20021827 87 Particle Systems  ...

  Chunk #14: 5928 chars
    Preview: Contents The Project Manager  ...................................................../uni200216 Preferences and Project!Settings  .........................../uni200217 Individual Preferences  and Settin...

  Chunk #6: 5742 chars
    Preview: 119 Tracking Nodes ..........................................................